# Fresh Kaggle session: v2 + full-gallery
Enable GPU and Internet; attach benchmark and both checkpoint model inputs. Run all six code cells in order. Requires updated repository source with --with-full-gallery. Full-gallery scoring is substantially slower than top-K. Default saves metrics/logs, not large similarity matrices.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO = Path("/kaggle/working/probably-works")

if not (REPO / ".git").exists():
    subprocess.run([
        "git", "clone",
        "--branch", "diagnostic",
        "--single-branch",
        "https://github.com/lqb464/probably-works.git",
        str(REPO),
    ], check=True)

os.chdir(REPO)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements.txt",
    "-r", "ex-of-ex/requirements.txt",
    "gdown",
], check=True)

subprocess.run(["git", "log", "-1", "--oneline"], check=True)

In [ ]:
import torch
import gdown

ROOT = Path(
    "/kaggle/input/datasets/hoanggv/tbps-benchmark/benchmark"
)
MODEL_DIR = Path(
    "/kaggle/input/models/hoanggv/clip-baseline/pytorch/default/1"
)
RSTP_BS256 = Path(
    "/kaggle/input/models/hoanggvo/tbps-clip-bs256/pytorch/default/1/best.pth"
)
RSTP_GDOWN = REPO / "rstp.pth"

assert torch.cuda.is_available(), "Chưa bật GPU trong Kaggle Settings."
assert ROOT.is_dir(), f"Thiếu dataset hoặc sai đường dẫn: {ROOT}"

for checkpoint in [
    MODEL_DIR / "icfg.pth",
    MODEL_DIR / "cuhk.pth",
    RSTP_BS256,
]:
    assert checkpoint.is_file(), f"Thiếu checkpoint: {checkpoint}"

if not RSTP_GDOWN.is_file() or RSTP_GDOWN.stat().st_size == 0:
    result = gdown.download(
        id="19ltsq0gyt7UJ31aJsgPIGRrxi-BvbAyy",
        output=str(RSTP_GDOWN),
        quiet=False,
    )
    assert result is not None, "Tải rstp.pth thất bại."

assert RSTP_GDOWN.is_file() and RSTP_GDOWN.stat().st_size > 0

print("GPU:", torch.cuda.get_device_name(0))
print("Đã có đủ dataset và 4 checkpoint.")

In [ ]:
import yaml

for name in ["rstp", "icfg", "cuhk"]:
    config_path = REPO / f"ex-of-ex/configs/{name}_validated.yaml"
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    protocol = config["validated"]

    assert protocol["enabled"]
    assert protocol["protocol_version"] == 2
    assert protocol["bootstrap_repetitions"] == 2000
    assert protocol["bootstrap_confidence"] == 0.95

subprocess.run([
    sys.executable, "-m", "unittest", "discover",
    "-s", "ex-of-ex", "-p", "tests*.py",
], cwd=REPO, check=True)
# Fail early if Kaggle still has the older source.
help_text = subprocess.check_output(
    [sys.executable, "ex-of-ex/run_experiments.py", "--help"],
    cwd=REPO, text=True,
)
assert "--with-full-gallery" in help_text, "Source is outdated: upload/publish the updated code first."


In [ ]:
jobs = [
    ("rstp", RSTP_GDOWN,            "rstp-gdown-all-experiments"),
    ("icfg", MODEL_DIR / "icfg.pth", "icfg-all-experiments"),
    ("cuhk", MODEL_DIR / "cuhk.pth", "cuhk-all-experiments"),
    ("rstp", RSTP_BS256,             "rstp-bs256-all-experiments"),
]

console_dir = REPO / "ex-of-ex/outputs/notebook_logs"
console_dir.mkdir(parents=True, exist_ok=True)

from datetime import datetime
batch_id = datetime.now().strftime("%Y%m%d-%H%M%S")
failed = []

for dataset, checkpoint, run_name in jobs:
    command = [
        sys.executable, "-u", "ex-of-ex/run_experiments.py",
        "--config", f"ex-of-ex/configs/{dataset}_validated.yaml",
        "--root_dir", str(ROOT),
        "--checkpoint", str(checkpoint),
        "--model-preset", "clip",
        "--with-full-gallery",
        "--run-name", run_name,
    ]

    log_path = console_dir / f"{batch_id}__{run_name}.log"
    print(f"\n===== {run_name} =====", flush=True)
    print(f"Console log: {log_path}", flush=True)

    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            cwd=REPO,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()

        return_code = process.wait()

    if return_code != 0:
        failed.append(run_name)
        print(f"LỖI: {run_name}; tiếp tục checkpoint kế tiếp.")

print("\nCác run lỗi:", failed if failed else "Không có")
assert not failed, "Có run thất bại. Xem console log tương ứng."

In [ ]:
import pandas as pd
from IPython.display import display

OUTPUTS = REPO / "ex-of-ex/outputs"
for run in sorted(OUTPUTS.glob("*/*__*-all-experiments")):
    print(f"\n===== {run.name} =====")
    for suite, filenames in {
        "validated_v2": [
            "setwise_summary.csv", "setwise_learned_summary.csv",
            "probe_primary_comparisons.csv", "reranking_selected_test.csv",
            "gating_summary.csv",
        ],
        "full_gallery": ["test_summary.csv"],
    }.items():
        results = run / suite
        if not (results / "summary.json").exists():
            print(f"{suite}: incomplete; inspect run.log")
        for filename in filenames:
            path = results / filename
            if path.exists():
                print(suite, filename)
                display(pd.read_csv(path))


In [ ]:
import shutil
from IPython.display import FileLink, display

archive = shutil.make_archive(
    "/kaggle/working/all-experiment-results",
    "zip",
    root_dir=str(REPO / "ex-of-ex"),
    base_dir="outputs",
)

os.chdir("/kaggle/working")
display(FileLink(Path(archive).name))